# 04 Model Training Starter

Project: **AI Support Operations SLA Breach and Priority Triage**

## Objective
Train baseline and taught model families for SLA breach classification. Optional: train a regression model for resolution_hours.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, precision_score, recall_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

path = Path("../01_Data/processed data/model_ready_support_sla_sample.csv")
if not path.exists():
    raise FileNotFoundError("Run the feature engineering notebook first.")

df = pd.read_csv(path, low_memory=False)
target = "sla_breached"
X = df.drop(columns=[target, "resolution_hours"], errors="ignore")
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category", "bool"]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


Numeric: ['hour_of_day', 'customer_tenure_months', 'contract_value_gbp', 'previous_tickets_90d', 'avg_sentiment_score', 'message_length', 'contains_urgent_keyword', 'contains_refund_keyword', 'agent_queue_length_at_submit', 'agent_experience_months', 'backlog_age_hours', 'first_response_minutes', 'reopened_last_90d', 'submitted_hour', 'submitted_dayofweek_num', 'submitted_month', 'is_weekend', 'contract_value_log1p', 'queue_pressure', 'negative_sentiment_flag', 'high_backlog_flag', 'message_word_count', 'message_has_deadline']
Categorical: ['customer_segment', 'uk_region', 'support_channel', 'product_area', 'issue_category', 'priority_initial', 'day_of_week', 'assigned_agent']


## Feature Review

Following feature engineering and leakage removal, the final modelling dataset contains 31 predictor variables. These include customer attributes, ticket characteristics, operational workload indicators, sentiment-based features, temporal features, and agent-related variables.

The dataset contains 23 numerical features and 8 categorical features. This mix of feature types requires appropriate preprocessing, including numerical scaling and categorical encoding, before model training.

Several features identified during exploratory analysis are expected to contribute strongly to predictive performance, particularly ticket priority, issue category, customer segment, queue pressure, backlog age, and response-related metrics. Together, these variables provide a comprehensive representation of ticket complexity, operational workload, and customer context at the point of prediction.


## Train/test split and preprocessing

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features)
    ]
)


## Data Splitting and Preprocessing

The dataset was divided into training and testing subsets using an 80:20 split. Stratified sampling was applied to preserve the original distribution of SLA breach outcomes across both datasets, ensuring representative evaluation results.

Separate preprocessing pipelines were created for numerical and categorical features. Numerical variables were imputed using median values and standardized using z-score normalization. Categorical variables were imputed using the most frequent category and transformed using one-hot encoding.

A column transformer was used to apply the appropriate preprocessing steps to each feature type. This approach ensures consistent data preparation during both training and prediction while reducing the risk of data leakage and improving reproducibility.


## Baseline and model family comparison

In [3]:
models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "decision_tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "random_forest": RandomForestClassifier(random_state=42, n_estimators=100, class_weight="balanced", n_jobs=-1),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
    "knn": KNeighborsClassifier(n_neighbors=7)
}

results = []
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X_test)[:, 1]
        roc = roc_auc_score(y_test, proba)
    else:
        roc = np.nan
    results.append({
        "model": name,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "roc_auc": roc
    })
    trained_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)


,model,precision,recall,f1,roc_auc
4,gradient_boosting,0.891979,0.913472,0.902597,0.961148
1,logistic_regression,0.928112,0.869660,0.897936,0.961869
3,random_forest,0.890988,0.904162,0.897527,0.955636
2,decision_tree,0.865464,0.919496,0.891662,0.946341
5,knn,0.831197,0.852136,0.841536,0.886114
0,dummy_most_frequent,0.608667,1.000000,0.756734,0.500000


## Model Comparison

Six classification models were evaluated using a common preprocessing pipeline and identical train-test split. Performance was assessed using Precision, Recall, F1-Score, and ROC-AUC.

The Gradient Boosting classifier achieved the strongest overall performance, producing an F1-Score of 0.903 and a ROC-AUC of 0.961. This indicates excellent discrimination between tickets that breach SLA targets and those that do not.

Logistic Regression and Random Forest also performed strongly, achieving F1-Scores close to 0.90 and ROC-AUC values above 0.95. The relatively small performance gap between these models suggests that the engineered features contain strong predictive information.

The Dummy Classifier served as a baseline by always predicting the majority class. Although it achieved high recall due to the class distribution, its ROC-AUC score of 0.50 confirms that it provides no meaningful predictive capability.

Overall, the results demonstrate that machine learning models can accurately identify tickets at risk of SLA breach, with Gradient Boosting providing the best balance between precision and recall.


## Inspect one model

In [4]:
best_name = results_df.iloc[0]["model"]
best_pipe = trained_pipelines[best_name]
preds = best_pipe.predict(X_test)

print("Best model by F1:", best_name)
print(classification_report(y_test, preds, zero_division=0))
print(confusion_matrix(y_test, preds))


Best model by F1: gradient_boosting
              precision    recall  f1-score   support

           0       0.86      0.83      0.84      1174
           1       0.89      0.91      0.90      1826

    accuracy                           0.88      3000
   macro avg       0.88      0.87      0.87      3000
weighted avg       0.88      0.88      0.88      3000

[[ 972  202]
 [ 158 1668]]


## Best Model Evaluation

The Gradient Boosting classifier achieved the highest F1-Score and was selected as the final classification model for detailed evaluation.

The model achieved an overall accuracy of 88%, with a precision of 89% and recall of 91% for the SLA breach class. The resulting F1-Score of 0.90 demonstrates a strong balance between correctly identifying SLA breaches and minimizing false alarms.

Analysis of the confusion matrix shows that the model correctly identified 1,668 SLA breaches and 972 non-breached tickets. Only 158 SLA breaches were missed, while 202 tickets were incorrectly flagged as potential breaches.

From a business perspective, the high recall rate is particularly valuable because it allows support teams to identify and intervene on the majority of high-risk tickets before SLA targets are missed. This capability could support proactive resource allocation, escalation management, and workload prioritisation strategies.

Overall, the model demonstrates strong predictive performance and provides a practical foundation for an SLA breach early-warning system.


## Optional regression extension

In [6]:
# Optional: predict resolution_hours using a regression target.
# Note: Do not use resolution_hours for the primary SLA breach classifier.
if "resolution_hours" in df.columns:
    y_reg = df["resolution_hours"]
    X_reg = df.drop(columns=["sla_breached", "resolution_hours"], errors="ignore")
    X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)
    reg_pipe = Pipeline(steps=[("preprocess", preprocess), ("model", LinearRegression())])
    reg_pipe.fit(X_train_r, y_train_r)
    reg_preds = reg_pipe.predict(X_test_r)
    print("MAE:", mean_absolute_error(y_test_r, reg_preds))
    print("RMSE:", np.sqrt(mean_squared_error(y_test_r, reg_preds)))
    print("R2:", r2_score(y_test_r, reg_preds))


MAE: 5.673757473958333
RMSE: 7.157872142863715
R2: 0.6329375060178061


## Optional Resolution Time Prediction

In addition to SLA breach classification, an optional regression model was developed to estimate ticket resolution time. A Linear Regression model was trained using the same feature set while excluding both the SLA breach target and the resolution time target from the predictor variables.

The model achieved a Mean Absolute Error (MAE) of 5.67 hours, indicating that predicted resolution times differed from actual values by approximately six hours on average. The Root Mean Squared Error (RMSE) was 7.16 hours, reflecting the presence of some larger prediction errors.

The model achieved an R² score of 0.633, meaning that approximately 63.3% of the variation in resolution time was explained by the available features. This demonstrates that ticket characteristics, customer information, workload indicators, and operational factors provide meaningful predictive information regarding resolution duration.

Although the primary objective of this project is SLA breach classification, the regression results suggest that future enhancements could include resolution-time forecasting and operational capacity planning tools.
